# 00 - GPU smoke test

Confirms that the single GKE node's L4 is visible to the container, that PyTorch
can use it, and roughly what throughput it delivers. Run every cell top to bottom.

`02-deploy-jupyter.sh` copies this notebook into `work/`, which is backed by a
PersistentVolumeClaim, so edits survive a pod restart.

## 1. Driver and device, straight from the node

In [ ]:
!nvidia-smi

In [ ]:
# Topology matrix. On a single-GPU node this is trivial, but the same command is
# how you confirm PCIe vs NVLink paths once there is more than one GPU per node.
!nvidia-smi topo -m

## 2. PyTorch sees the GPU

In [ ]:
import torch

print('torch          :', torch.__version__)
print('cuda available :', torch.cuda.is_available())
print('cuda runtime   :', torch.version.cuda)
print('device count   :', torch.cuda.device_count())

assert torch.cuda.is_available(), 'No CUDA device. Check the pod has nvidia.com/gpu: 1.'

props = torch.cuda.get_device_properties(0)
print()
print('name           :', props.name)
print('capability     : sm_%d%d' % (props.major, props.minor))
print('total memory   : %.1f GiB' % (props.total_memory / 1024**3))
print('SM count       :', props.multi_processor_count)
print('bf16 supported :', torch.cuda.is_bf16_supported())

## 3. Matmul throughput

A crude but honest check that the GPU is actually doing work. An L4 is rated at
roughly 120 TFLOP/s dense bf16; a real measurement in the 60-100 range is normal
for this size, and anything near 1 means you are silently running on CPU.

In [ ]:
import time
import torch

n, iters = 8192, 50
a = torch.randn(n, n, device='cuda', dtype=torch.bfloat16)
b = torch.randn(n, n, device='cuda', dtype=torch.bfloat16)

for _ in range(10):          # warm up: first calls include kernel autotuning
    a @ b
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(iters):
    a @ b
torch.cuda.synchronize()
elapsed = time.perf_counter() - start

flops = 2 * n**3 * iters     # one multiply-add per output element per k
print('%.1f ms/matmul' % (elapsed / iters * 1e3))
print('%.1f TFLOP/s bf16' % (flops / elapsed / 1e12))

## 4. NCCL initialises

Single process, single rank, so this measures nothing about the network. It only
proves the NCCL library loads and the process group forms, which is the thing that
breaks first when you later scale to multiple nodes.

In [ ]:
import os
import torch
import torch.distributed as dist

os.environ.setdefault('MASTER_ADDR', '127.0.0.1')
os.environ.setdefault('MASTER_PORT', '29500')
os.environ.setdefault('RANK', '0')
os.environ.setdefault('WORLD_SIZE', '1')

if not dist.is_initialized():
    dist.init_process_group(backend='nccl')

torch.cuda.set_device(0)
t = torch.ones(1024, device='cuda')
dist.all_reduce(t)
torch.cuda.synchronize()

print('world size :', dist.get_world_size())
print('all_reduce :', t[0].item(), '(expected 1.0 at world size 1)')

dist.destroy_process_group()

## 5. Where the pod is running

Useful once there is more than one node and you need to know which one you landed on.

In [ ]:
import socket
import subprocess

print('pod hostname :', socket.gethostname())
print()
print(subprocess.run(['bash', '-lc', 'cat /proc/cpuinfo | grep -c ^processor'],
                     capture_output=True, text=True).stdout.strip(), 'vCPU visible')
print(subprocess.run(['bash', '-lc', "free -g | awk '/Mem:/ {print $2}'"],
                     capture_output=True, text=True).stdout.strip(), 'GiB RAM visible')